# 📒 Portfolio Agent – Notebook (Corrigido)

In [ ]:
from __future__ import annotations
%pip -q install yfinance pandas numpy scipy scikit-learn plotly reportlab kaleido tenacity

In [ ]:

import math, os, datetime
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

import plotly.graph_objs as go
import plotly.io as pio

from tenacity import retry, stop_after_attempt, wait_exponential

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.pdfgen import canvas

import warnings
warnings.filterwarnings("ignore")


In [ ]:

@dataclass
class DataConfig:
    tickers: List[str]
    start: str
    end: str
    interval: str = "1d"

@dataclass
class ModelConfig:
    model_type: str = "ridge"  # 'ridge' | 'random_forest'
    lookback_days: int = 252
    horizon_days: int = 21
    feature_windows: Dict[str, int] = field(default_factory=lambda: {
        "ret_21": 21, "ret_63": 63, "mom_126": 126, "vol_21": 21
    })

@dataclass
class OptimizeConfig:
    long_only: bool = True
    max_weight: float = 0.2
    risk_aversion: float = 5.0
    rebalance: str = "M"
    transaction_cost_bps: float = 2.0

@dataclass
class ReportConfig:
    output_pdf: str = "relatorio_portfolio.pdf"
    title: str = "Relatório de Portfólio – Agente Preditivo"

@dataclass
class PipelineConfig:
    data: DataConfig
    model: ModelConfig = field(default_factory=ModelConfig)
    optimize: OptimizeConfig = field(default_factory=OptimizeConfig)
    report: ReportConfig = field(default_factory=ReportConfig)


In [ ]:

import yfinance as yf

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=8))
def download_ohlc(ticker: str, start: str, end: str, interval: str="1d") -> pd.DataFrame:
    df = yf.download(ticker, start=start, end=end, interval=interval, auto_adjust=True, progress=False)
    if not isinstance(df, pd.DataFrame) or df.empty:
        raise RuntimeError(f"Sem dados para {ticker}.")
    df = df.rename(columns=str.title)
    df["Ticker"] = ticker
    return df

def load_prices(tickers: List[str], start: str, end: str, interval: str="1d") -> pd.DataFrame:
    frames = []
    for t in tickers:
        df = download_ohlc(t, start, end, interval)
        frames.append(df[["Close"]].assign(Ticker=t))
    allc = pd.concat(frames).reset_index().rename(columns={"Date":"Date"})
    prices = allc.pivot(index="Date", columns="Ticker", values="Close").sort_index()
    return prices.dropna(how="all")


In [ ]:

def rolling_return(prices: pd.DataFrame, window: int) -> pd.DataFrame:
    return prices.pct_change(window)

def momentum(prices: pd.DataFrame, window: int) -> pd.DataFrame:
    return prices / prices.shift(window) - 1.0

def volatility(returns: pd.DataFrame, window: int) -> pd.DataFrame:
    return returns.rolling(window).std()

def make_feature_panel(prices: pd.DataFrame, feature_windows: dict) -> dict:
    rets = prices.pct_change().dropna()
    feats = {}
    if "ret_21" in feature_windows:
        feats["ret_21"] = rolling_return(prices, feature_windows["ret_21"])
    if "ret_63" in feature_windows:
        feats["ret_63"] = rolling_return(prices, feature_windows["ret_63"])
    if "mom_126" in feature_windows:
        feats["mom_126"] = momentum(prices, feature_windows["mom_126"])
    if "vol_21" in feature_windows:
        feats["vol_21"] = volatility(rets, feature_windows["vol_21"])
    return feats


In [ ]:

def _prepare_xy(feats: Dict[str, pd.DataFrame], target: pd.DataFrame, as_of_date: pd.Timestamp, lookback_days: int):
    X_list, y_list = [], []
    dates = target.index[target.index < as_of_date][-lookback_days:]
    tickers = target.columns
    for dt in dates:
        row = []
        ok = True
        for name, df in feats.items():
            if dt not in df.index:
                ok = False; break
            row.append(df.loc[dt, tickers].values.reshape(1,-1))
        if not ok: 
            continue
        X = np.concatenate(row, axis=1)
        y = target.loc[dt, tickers].values.reshape(1, -1)
        X_list.append(X); y_list.append(y)
    if not X_list:
        raise ValueError("Sem dados suficientes para treinar.")
    X = np.vstack(X_list)
    y = np.vstack(y_list).mean(axis=1)
    return X, y

def train_predict_expected_returns(feats: Dict[str, pd.DataFrame],
                                   future_returns: pd.DataFrame,
                                   as_of_date: pd.Timestamp,
                                   lookback_days: int,
                                   model_type: str="ridge") -> pd.Series:
    X, y = _prepare_xy(feats, future_returns, as_of_date, lookback_days)
    if model_type == "ridge":
        mdl = Ridge(alpha=10.0)
    elif model_type == "random_forest":
        mdl = RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42)
    else:
        raise ValueError("model_type inválido.")
    mdl.fit(X, y)

    tickers = future_returns.columns
    row = []
    for name, df in feats.items():
        if as_of_date not in df.index:
            raise ValueError(f"Features indisponíveis para {as_of_date}.")
        row.append(df.loc[as_of_date, tickers].values.reshape(1,-1))
    X_t = np.concatenate(row, axis=1)

    mean_pred = float(mdl.predict(X_t)[0])
    mom = feats.get("mom_126")
    if mom is not None and as_of_date in mom.index:
        rel = mom.loc[as_of_date]
        rel = (rel - rel.mean()) / (rel.std() + 1e-9)
        pred_each = mean_pred + 0.5 * rel
    else:
        pred_each = pd.Series(mean_pred, index=tickers)
    return pd.Series(pred_each, index=tickers).astype(float)


In [ ]:

def optimize_mvo(expected: pd.Series,
                 cov: pd.DataFrame,
                 long_only: bool = True,
                 max_weight: float = 0.2) -> pd.Series:
    mu = expected.values
    Sigma = cov.values + 1e-6 * np.eye(len(mu))
    invS = np.linalg.pinv(Sigma)
    w = invS @ mu
    if long_only:
        w = np.maximum(w, 0.0)
    w = w / (w.sum() + 1e-12)
    if max_weight is not None:
        w = np.minimum(w, max_weight)
        s = w.sum()
        if s <= 0:
            w = np.ones_like(w)/len(w)
        else:
            w = w / s
    return pd.Series(w, index=expected.index)


In [ ]:

def backtest(prices: pd.DataFrame,
             expected_next: pd.DataFrame,
             cov_roll: dict,
             rebalance_dates,
             long_only=True,
             max_weight=0.2,
             tc_bps=2.0):
    equity_curve = pd.Series(dtype=float)
    weights_hist = {}
    nav = 1.0
    last_w = None

    for dt in rebalance_dates:
        mu = expected_next.loc[dt].dropna()
        if dt not in cov_roll: 
            continue
        cov = cov_roll[dt].loc[mu.index, mu.index]
        w = optimize_mvo(mu, cov, long_only=long_only, max_weight=max_weight)
        if last_w is None:
            tc = 0.0
        else:
            tc = (np.abs(w - last_w).sum()) * (tc_bps / 10000.0)
        last_w = w
        weights_hist[dt] = w

        if dt not in prices.index:
            continue
        idx = prices.index
        start = idx.get_loc(dt)
        end = min(start + 21, len(idx)-1)
        window = idx[start:end+1]
        rets = prices.loc[window, w.index].pct_change().fillna(0.0)
        port_rets = (rets @ w).clip(-0.5, 0.5)
        port_rets.iloc[0] -= tc
        growth = (1 + port_rets).cumprod()
        nav *= growth.iloc[-1]
        equity_curve = pd.concat([equity_curve, pd.Series([nav], index=[window[-1]])])

    return equity_curve.sort_index(), weights_hist


In [ ]:

def run_pipeline(cfg: "PipelineConfig"):
    prices = load_prices(cfg.data.tickers, cfg.data.start, cfg.data.end, cfg.data.interval)
    daily_rets = prices.pct_change().dropna()

    feats = make_feature_panel(prices, cfg.model.feature_windows)
    fut = prices.pct_change(cfg.model.horizon_days).shift(-cfg.model.horizon_days)

    rebalance = prices.resample(cfg.optimize.rebalance).last().index
    rebalance = [d for d in rebalance if d in prices.index]

    cov_roll = {}
    expected_next = {}
    for dt in rebalance:
        hist = daily_rets.loc[:dt].tail(cfg.model.lookback_days)
        cov_roll[dt] = hist.cov()
        mu_pred = train_predict_expected_returns(feats, fut, as_of_date=dt,
                                                 lookback_days=cfg.model.lookback_days,
                                                 model_type=cfg.model.model_type)
        expected_next[dt] = mu_pred
    expected_next = pd.DataFrame(expected_next).T

    curve, weights_hist = backtest(prices,
                                   expected_next=expected_next,
                                   cov_roll=cov_roll,
                                   rebalance_dates=rebalance,
                                   long_only=cfg.optimize.long_only,
                                   max_weight=cfg.optimize.max_weight,
                                   tc_bps=cfg.optimize.transaction_cost_bps)
    port_rets = curve.pct_change().dropna()
    ann_factor = 252
    mu_ann = (1 + port_rets.mean())**ann_factor - 1 if len(port_rets)>0 else 0.0
    vol_ann = port_rets.std() * np.sqrt(ann_factor) if len(port_rets)>0 else 0.0
    dd = (curve / curve.cummax() - 1).min() if len(curve)>0 else 0.0
    sharpe = (mu_ann - 0.0) / (vol_ann + 1e-12) if vol_ann>0 else 0.0
    metrics = {
        "Retorno anualizado": f"{mu_ann:.2%}",
        "Vol anualizada": f"{vol_ann:.2%}",
        "Sharpe": f"{sharpe:.2f}",
        "Max Drawdown": f"{dd:.2%}"
    }

    fig_curve = go.Figure()
    fig_curve.add_trace(go.Scatter(x=curve.index, y=curve.values, mode="lines", name="Portfólio"))
    fig_curve.update_layout(title="Evolução do Portfólio (NAV)", xaxis_title="Data", yaxis_title="NAV")

    dates = sorted(weights_hist.keys())
    tickers = sorted({t for w in weights_hist.values() for t in w.index}) if dates else []
    dfw = pd.DataFrame(index=dates, columns=tickers, data=0.0) if dates else pd.DataFrame()
    for dt, w in weights_hist.items():
        dfw.loc[dt, w.index] = w.values
    fig_w = go.Figure()
    if not dfw.empty:
        for t in dfw.columns:
            fig_w.add_trace(go.Bar(x=dfw.index, y=dfw[t], name=t))
        fig_w.update_layout(barmode="stack", title="Pesos por Rebalance", xaxis_title="Data", yaxis_title="Peso")

    return {"prices": prices, "equity_curve": curve, "weights_hist": weights_hist, "metrics": metrics,
            "fig_curve": fig_curve, "fig_weights": fig_w}


In [ ]:

def build_html_report(output_html: str, title: str, metrics: dict, fig_curve=None, fig_weights=None):
    parts = []
    parts.append(f"""<!doctype html><html lang='pt-br'><head>
<meta charset='utf-8'/><meta name='viewport' content='width=device-width, initial-scale=1'/>
<title>{title}</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>body{font-family:system-ui,-apple-system,'Segoe UI',Roboto,Ubuntu,'Helvetica Neue',Arial;margin:24px;}
.card{border:1px solid #e5e7eb;border-radius:12px;padding:16px;margin-bottom:16px;box-shadow:0 1px 2px rgba(0,0,0,.04);}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;}
.stat{background:#f9fafb;padding:10px;border-radius:10px;}</style></head><body>
<h1>📈 {title}</h1><div class='card'><h3>Métricas</h3><div class='grid'>""")
    parts.append(''.join([f"<div class='stat'><b>{k}</b><br>{v}</div>" for k,v in metrics.items()]))
    parts.append("</div></div>")
    if fig_curve is not None:
        parts.append("<div class='card'><h3>Equity Curve</h3>")
        parts.append(pio.to_html(fig_curve, include_plotlyjs=False, full_html=False)); parts.append("</div>")
    if fig_weights is not None:
        parts.append("<div class='card'><h3>Pesos por Rebalance</h3>")
        parts.append(pio.to_html(fig_weights, include_plotlyjs=False, full_html=False)); parts.append("</div>")
    parts.append(f"<footer>Gerado em {datetime.date.today()}</footer></body></html>")
    with open(output_html, "w", encoding="utf-8") as f:
        f.write('\n'.join(parts))
    return output_html


In [ ]:

# Parâmetros rápidos
TICKERS = "AAPL MSFT AMZN GOOGL META".split()
START = "2018-01-01"
END = datetime.date.today().isoformat()
MODEL = "ridge"
LOOKBACK = 252
HORIZON = 21
REBALANCE = "M"
MAX_WEIGHT = 0.2
TC_BPS = 2.0

cfg = PipelineConfig(
    data=DataConfig(tickers=TICKERS, start=START, end=END, interval="1d"),
    model=ModelConfig(model_type=MODEL, lookback_days=LOOKBACK, horizon_days=HORIZON),
    optimize=OptimizeConfig(long_only=True, max_weight=MAX_WEIGHT, rebalance=REBALANCE, transaction_cost_bps=TC_BPS),
    report=ReportConfig(output_pdf="relatorio_portfolio.pdf")
)
cfg


In [ ]:

results = run_pipeline(cfg)
results["metrics"]


In [ ]:

results["fig_curve"].show()


In [ ]:

results["fig_weights"].show()


In [ ]:

build_html_report("report_interativo.html", "Relatório de Portfólio – Agente Preditivo", results["metrics"], results["fig_curve"], results["fig_weights"])
